# Force Computation in Numerical Cosmology

---

## Outline

1. Particle–Particle (PP)  
2. Particle–Mesh (PM)  
3. Particle–Particle–Particle–Mesh (P³M)  
4. Tree  
5. Tree–PM  
6. Fast Multipole Method (FMM)  
7. Fast Multipole Method–PM (FMM–PM)  

---

<div style="text-align:center;">
  <img src="forces.png" alt="Forces Plot" style="width:auto; height:auto;">
</div>

## Particle–Particle (PP)

---

**Principle**  
Compute forces by direct summation over all particle pairs:  
– $\mathbf{F}_i = \sum_{j\neq i} G\,\frac{m_i m_j}{|\mathbf{r}_i - \mathbf{r}_j|^2}\,\hat{\mathbf{r}}_{ij}$

**Complexity**  
– $O(N^2)$ operations for $N$ particles.

**Advantages**  
– Exact force evaluation at all separations.  

**Limitations**  
– Prohibitive cost for $N\gtrsim10^5$.  

## Particle–Mesh (PM)

---

**Principle**  
1. Deposit mass onto a fixed grid.  
2. Solve Poisson’s equation via Fast Fourier Transform (FFT).  
3. Interpolate field back to particle positions.

**Complexity**  
– $O(N + M\log M)$ for $M$ grid cells.

**Advantages**  
– Efficient for large $N$.  
– Periodic boundary conditions handled naturally.

**Limitations**  
– Poor resolution at sub–cell scales.  
– Force anisotropy from grid.

## Particle–Particle–Particle–Mesh (P³M)

---

**Principle**  
Combine PP and PM:  
- Long-range forces via PM.  
- Short-range corrections via PP within a cutoff radius.

**Complexity**  
– $O(N\log N + N_{\rm short}^2)$, where $N_{\rm short}$ is local neighbours.

**Advantages**  
– Improved small-scale accuracy.  
– Retains PM efficiency for long range.

**Limitations**  
– Load imbalance in dense regions.  
– Still $O(N_{\rm short}^2)$ worst case locally.

## Tree Method

---

**Principle**  
Organise particles in an octree (3D) or quadtree (2D).  
Approximate distant clusters by their multipole moments.

**Complexity**  
– $O(N\log N)$ on average.

**Advantages**  
– Adaptive resolution.  
– Good for inhomogeneous distributions.

**Limitations**  
– Tree build and traversal overhead.  
– Multipole truncation error.

## Tree–PM

---

**Principle**  
Hybrid of Tree and PM:  
- Long-range via PM grid.  
- Short-range via tree walk within a given radius.

**Complexity**  
– $O(N\log N + M\log M)$

**Advantages**  
– Combines PM efficiency with tree adaptivity.  
– Balanced workload for both regimes.

**Limitations**  
– Two data structures to maintain.  
– Parameter tuning for split radius.

## Fast Multipole Method (FMM)

---

**Principle**  
Hierarchical multipole expansions for both near and far fields.  
Allows particle groups to interact via multipoles directly.

**Complexity**  
– $O(N)$ with careful implementation.

**Advantages**  
– Linear scaling.  
– Controlled error via expansion order.

**Limitations**  
– Complex implementation.  
– Higher constant factors.

## Fast Multipole Method–PM (FMM–PM)

---
    
**Principle**  
Combine FMM for short scales with PM for long scales.  
- Long-range: FFT on grid.  
- Short-range: FMM within interaction radius.

**Complexity**  
– $O(N + M\log M)$

**Advantages**  
– Linear to quasi-linear scaling.  
– High accuracy at all scales.

**Limitations**  
– Implementation complexity.  
– Memory overhead for dual structures.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Load the artistic background image
bg_path = 'cluster.png'
bg = mpimg.imread(bg_path)

# Domain and cluster parameters (for overlay circle)
domain_size = 1.0
circle_center = (0.5, 0.5)
circle_radius = 0.2
theta = np.linspace(0, 2 * np.pi, 200)
circle_x = circle_center[0] + circle_radius * np.cos(theta)
circle_y = circle_center[1] + circle_radius * np.sin(theta)

# Create three-panel figure
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Common plotting function for grid
def plot_grid(ax, lines, color='white', linewidth=0.8):
    for (xs, ys) in lines:
        ax.plot(xs, ys, color=color, linewidth=linewidth, zorder=2)

# Panel 1: Fixed PM mesh
ax = axes[0]
ax.imshow(bg, extent=(0, domain_size, 0, domain_size), origin='lower')
# Uniform grid
spacing = 0.1
lines = []
for x in np.arange(0, domain_size + spacing, spacing):
    lines.append(([x, x], [0, domain_size]))
for y in np.arange(0, domain_size + spacing, spacing):
    lines.append(([0, domain_size], [y, y]))
plot_grid(ax, lines)
# Overlay cluster circle
ax.plot(circle_x, circle_y, color='white', linewidth=1.5, zorder=3)
ax.set_title('PM: Fixed Mesh', color='white')
ax.axis('off')

# Panel 2: Adaptive PM
ax = axes[1]
ax.imshow(bg, extent=(0, domain_size, 0, domain_size), origin='lower')
# Coarse grid
lines_coarse = []
spacing_coarse = 0.1
for x in np.arange(0, domain_size + spacing_coarse, spacing_coarse):
    lines_coarse.append(([x, x], [0, domain_size]))
for y in np.arange(0, domain_size + spacing_coarse, spacing_coarse):
    lines_coarse.append(([0, domain_size], [y, y]))
plot_grid(ax, lines_coarse)
# AMR patch
patch_min, patch_max = 0.3, 0.7
# Outline
ax.plot([patch_min, patch_max, patch_max, patch_min, patch_min],
        [patch_min, patch_min, patch_max, patch_max, patch_min],
        color='cyan', linewidth=1.5, zorder=4)
# Fine grid inside patch
lines_fine = []
spacing_fine = 0.05
for x in np.arange(patch_min, patch_max + spacing_fine, spacing_fine):
    lines_fine.append(([x, x], [patch_min, patch_max]))
for y in np.arange(patch_min, patch_max + spacing_fine, spacing_fine):
    lines_fine.append(([patch_min, patch_max], [y, y]))
plot_grid(ax, lines_fine, color='cyan', linewidth=0.8)
# Cluster circle
ax.plot(circle_x, circle_y, color='white', linewidth=1.5, zorder=3)
ax.set_title('Adaptive PM: AMR Patch', color='white')
ax.axis('off')

# Panel 3: Tree (Quadtree)
ax = axes[2]
ax.imshow(bg, extent=(0, domain_size, 0, domain_size), origin='lower')
# Quadtree decomposition
def subdivide(cell, depth, max_depth):
    x0, y0, x1, y1 = cell
    cx, cy = circle_center
    closest_x = np.clip(cx, x0, x1)
    closest_y = np.clip(cy, y0, y1)
    dist = np.hypot(closest_x - cx, closest_y - cy)
    if dist < circle_radius and depth < max_depth:
        mx, my = 0.5*(x0 + x1), 0.5*(y0 + y1)
        cells = []
        for nx0, ny0, nx1, ny1 in [
            (x0, y0, mx, my), (mx, y0, x1, my),
            (x0, my, mx, y1), (mx, my, x1, y1)
        ]:
            cells.extend(subdivide((nx0, ny0, nx1, ny1), depth+1, max_depth))
        return cells
    else:
        return [cell]

cells = subdivide((0, 0, domain_size, domain_size), 0, max_depth=3)
for (x0, y0, x1, y1) in cells:
    ax.plot([x0, x1, x1, x0, x0],
            [y0, y0, y1, y1, y0],
            color='white', linewidth=0.8, zorder=2)
# Cluster circle
ax.plot(circle_x, circle_y, color='white', linewidth=1.5, zorder=3)
ax.set_title('Tree: Quadtree', color='white')
ax.axis('off')

plt.tight_layout()
plt.savefig("forces.png")
plt.show()

# Contribution of Relativistic Species to the Gravitational Potential

---

## Motivation

- Standard N-body simulations include only cold dark matter (CDM) and baryons.  
- Photon and neutrino perturbations alter the large-scale potential at the percent level.  
- Accurate modelling of CMB and large-scale structure requires their inclusion.  

---

## COSIRA Method Overview

- **COSmological Initial Conditions including Relativistic species in N-body simulations (COSIRA)**  
- Realize linear density field of relativistic species from a Boltzmann code power spectrum.  
- Impose Fourier phases from the existing N-body initial conditions.  
- Compute potential contribution via Poisson equation and add to total.  

---

## Realizing the Relativistic Density Field

1. Compute linear power spectrum $P_{\rm rel}(k,z)$ for photons + neutrinos using CLASS or CAMB.  
2. Generate Fourier modes  
   $$
   \delta_{\rm rel}(\mathbf{k},z) = \sqrt{P_{\rm rel}(k,z)}\;\exp\bigl[i\,\varphi_{\rm CDM}(\mathbf{k})\bigr],
   $$  
   where $\varphi_{\rm CDM}(\mathbf{k})$ are the phases from the CDM initial realization.  

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import data, transform

# Load two built-in grayscale images
img1 = data.camera()   # Image for amplitudes
img2 = data.moon()     # Image for phases

# Ensure both images have the same shape
if img1.shape != img2.shape:
    img2 = transform.resize(img2, img1.shape, anti_aliasing=True)
    img2 = (img2 * 255).astype(np.uint8)

# Compute the 2D Fourier transforms
fft1 = np.fft.fft2(img1)
fft2 = np.fft.fft2(img2)

# Extract amplitude (magnitude) from img1 and phase from img2
amp1   = np.abs(fft1)
phase2 = np.angle(fft2)

# Combine amplitude of img1 with phase of img2
combined_fft = amp1 * np.exp(1j * phase2)
combined     = np.fft.ifft2(combined_fft)
combined     = np.real(combined)

# Normalize combined image to the 0–255 range, convert to uint8
combined_norm = (combined - combined.min()) / (combined.max() - combined.min())
combined_norm = (combined_norm * 255).astype(np.uint8)

# Display the original and combined images
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(img1, cmap='gray')
axes[0].set_title('Image 1 (Amplitudes)')
axes[0].axis('off')

axes[1].imshow(img2, cmap='gray')
axes[1].set_title('Image 2 (Phases)')
axes[1].axis('off')

axes[2].imshow(combined_norm, cmap='gray')
axes[2].set_title('Combined (Amp₁ + Ph₂)')
axes[2].axis('off')

plt.tight_layout()
plt.savefig("phases_and_amplitudes.png")
plt.show()

<div style="text-align:center;">
  <img src="phases_and_amplitudes.png" alt="COSIRA Plot" style="width:100%; height:auto;">
</div>

## From Density to Potential

- Solve Poisson’s equation in Fourier space:  
  $$\Phi_{\rm rel}(\mathbf{k},z) 
    = -\,4\pi G\,a^2(z)\,\bar\rho_{\rm rel}(z)\;\frac{\delta_{\rm rel}(\mathbf{k},z)}{k^2}.$$
- Inverse Fourier transform to obtain $\Phi_{\rm rel}(\mathbf{x})$ on the grid.  
- Add $\Phi_{\rm rel}$ to the long-range potential used in the force solver.  

---

## Method and Implementation (Section 2 of Tram et al. 2019)

**Equations of Motion in N-body Gauge**  
$$
\dot\delta_{\rm Nb} + \nabla\!\cdot\!v_{\rm Nb} = 0,
\quad
(\partial_\tau + \mathcal H)\,v_{\rm Nb} = -\nabla\phi + \nabla\gamma_{\rm Nb}.
$$  
Here $\gamma_{\rm Nb}$ is the relativistic correction that vanishes without radiation.

**Split of the Total Potential**  
$$
\phi - \gamma_{\rm Nb} \;\equiv\; \phi_{\rm sim} + \phi_{\rm GR},
\quad
\nabla^2\phi_{\rm GR} = 4\pi G\,a^2\,\delta\rho_{\rm GR}.
$$

**Components of $\delta\rho_{\rm GR}$**  
- $\delta\rho_\gamma$ from photons  
- $\delta\rho_\nu$ from neutrinos  
- $\delta\rho_{\rm metric}$ defined by  
  $\nabla^2\gamma_{\rm Nb} = -4\pi G\,a^2\,\delta\rho_{\rm metric}.$

**Realisation Procedure**  
1. **CLASS outputs:**  
   - $\delta\rho_\gamma(k),\,\delta\rho_\nu(k)$  
   - Trace-free metric mode $\dot H^T_{\rm Nb}(k)$  
   - Total anisotropic stress $\Sigma(k)$  
2. **Compute $\gamma(k)$:**  
   $$
   k^2\gamma = -\,(\partial_\tau + \mathcal H)\,\dot H^T_{\rm Nb} \;+\; 8\pi G a^2\,\Sigma.
   $$

3. **Gauge-transform densities:**  
   $$
   \delta\rho^{\rm Nb}_\alpha
   = \delta\rho^{S/N}_\alpha
     + 3\mathcal H\,(1+w_\alpha)\,
       \frac{\theta_{\rm tot}}{k^2}\,\bar\rho_\alpha.
   $$
4. **Sum** $\delta\rho_{\rm GR}(k)=\delta\rho_\gamma^{\rm Nb}+\delta\rho_\nu^{\rm Nb}+\delta\rho_{\rm metric}(k)$.  
5. **Grid realisation:** inverse FFT $\to$ $\delta\rho_{\rm GR}(x)$.  
6. **Force injection:** solve Poisson for $\phi_{\rm GR}$, apply $\nabla\phi_{\rm GR}$ to particles at each timestep.

<div style="text-align:center;">
  <img src="density_1x3.png" alt="CDE Plot" style="width:100%; height:auto;">
</div>

---

## Validation and Performance

- **Validation:** compare large-scale potential power with linear theory.  
- **Overhead:** one additional FFT pair per redshift output.  
- **Memory:** one complex grid of size $N_{\rm grid}^3$.  
- **Accuracy:** per-mille level on linear scales for neutrino masses up to ∼0.5 eV.  

---

## Reference

T. Tram, J. Brandbyge, J. Dakin & S. Hannestad,  
“Fully relativistic treatment of light neutrinos in N-body simulations”,  
JCAP 03 (2019) 022, DOI: 10.1088/1475-7516/2019/03/022.  

# Hydrodynamic Solvers in Numerical Cosmology

---

1. SPH - GADGET-2
2. MFM
3. Moving Mesh - AREPO
4. AMR RAMSES

<div style="text-align:center;">
  <img src="hydro.png" alt="Hydro Plot" style="width:80%; height:auto;">
</div>

## Smoothed Particle Hydrodynamics (SPH) – GADGET-2

---

**Principle**  
- Lagrangian method: fluid represented by particles.  
- Physical quantities smoothed over neighbours via a kernel function.  
- Artificial viscosity handles shocks.

**Computational Cost**  
- $\mathcal O(N \log N)$ for neighbour search with tree structures.

**Advantages**  
- Exact mass conservation.  
- Natural adaptivity to density.  

**Limitations**  
- Poor resolution of contact discontinuities (surface tension effect).  
- Noise in density and pressure estimates.  
- Reliance on artificial viscosity for shock capturing.  

## Meshless Finite Mass (MFM)

---

**Principle**  
- Particles carry mass like SPH but solve Riemann problems between effective “faces.”  
- Godunov-type finite-volume update without a fixed grid.

**Computational Cost**  
- $\mathcal O(N \log N)$ for neighbour finding and flux evaluation.

**Advantages**  
- Sharp shock resolution without artificial viscosity.  
- Good angular momentum conservation.  
- Minimal advection error.  

**Limitations**  
- Sensitive to particle disorder.  
- Kernel and face reconstruction add complexity.  
- Implementation is more involved than SPH.  

## Moving Mesh – AREPO

---

**Principle**  
- Voronoi tessellation defines a dynamic mesh that moves with the flow.  
- Finite-volume Godunov solver across cell faces.

**Computational Cost**  
- $\mathcal O(N \log N)$ for mesh construction and neighbour search.

**Advantages**  
- Low advection error and accurate Galilean invariance.  
- Sharp and accurate shock capturing.  
- Automatic adaptivity and quasi-Lagrangian behaviour.  

**Limitations**  
- Overhead of dynamic mesh generation each timestep.  
- Complexity of mesh maintenance and parallel communication.  
- Potential for mesh distortion in highly turbulent flows.  

## Adaptive Mesh Refinement – RAMSES

---

**Principle**  
- Eulerian grid with octree AMR: refine cells where higher resolution is needed.  
- Second-order Godunov scheme with constrained transport for magnetohydrodynamics.

**Computational Cost**  
- $\mathcal O(N_{\rm cells} \log N_{\rm cells})$, where $N_{\rm cells}$ grows in dense regions.

**Advantages**  
- High spatial resolution in regions of interest.  
- Robust shock capturing and well-tested solvers.  
- Flexible refinement criteria (density, Jeans length, etc.).  

**Limitations**  
- Grid-induced anisotropy at coarse–fine interfaces.  
- Memory overhead from storing tree metadata.  
- Less natural for highly supersonic, filamentary flows.  